<a href="https://colab.research.google.com/github/deva252/PYTHON-CASE-STUDY/blob/main/DEVAPRIYA_Case_Study_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Case Study Assessment — NYC Airbnb Data Preprocessing
### Student Name:DEVA PRIYA B
### Date:06-09-2026

This notebook is your submission template. Fill in each section — do not delete the headers. Use markdown cells for your written justifications, right next to the code that supports them.

In [ ]:
import pandas as pd
url = "https://raw.githubusercontent.com/erkansirin78/datasets/master/AB_NYC_2019.csv"
df = pd.read_csv(url)
df.head()

---
## Part 1 — Data Understanding & Quality Audit

In [ ]:
# TODO: shape, dtypes, missing value summary
print("Shape:", df.shape)

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
missing_summary = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing %": (df.isnull().mean() * 100).round(2)
})

print(missing_summary[missing_summary["Missing Count"] > 0])

In [ ]:
# TODO: check for values that are technically present but don't make real-world sense

print("Price <= 0:", (df["price"] <= 0).sum())
print("Minimum nights > 365:", (df["minimum_nights"] > 365).sum())
print("Minimum nights > 1095:", (df["minimum_nights"] > 1095).sum())

print("Negative number of reviews:", (df["number_of_reviews"] < 0).sum())
print("Reviews per month < 0:", (df["reviews_per_month"] < 0).sum())
print("Availability outside 0-365:",
      ((df["availability_365"] < 0) | (df["availability_365"] > 365)).sum())

print("\nNumeric summary:")
display(df.describe())

In [ ]:
# TODO: duplicate check
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

**Written notes — what did you find, and what looks suspicious?**

The dataset contains 48,895 rows and 16 columns. The data contains numerical, categorical and date-related variables.

The main missing values occur in name, host_name, last_review and reviews_per_month. The last_review and reviews_per_month columns each contain 10,052 missing values.

The dataset also contains suspicious values. Price contains zero-dollar listings, which do not represent a valid price for a price-prediction task. minimum_nights has extreme values, including values greater than 365 days and a maximum of 1,250 days. These values are unusual for a short-term rental listing and should be investigated.

The maximum price is $10,000 per night. Although this is an extreme statistical outlier, it may represent a genuine luxury listing rather than a data-entry error, so it should not automatically be deleted.

The dataset has no duplicated complete rows.

---
## Part 2 — Missing Value Diagnosis & Treatment

In [ ]:
# TODO: investigate missingness pattern(s) across columns
missing_summary = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage": (df.isnull().mean() * 100).round(2)
})

display(
    missing_summary[missing_summary["missing_count"] > 0]
    .sort_values("missing_count", ascending=False)
)

In [ ]:
review_missing = df["reviews_per_month"].isna()

print(pd.crosstab(
    review_missing,
    df["number_of_reviews"].eq(0),
    normalize="index"
))

print("\nRows with missing reviews_per_month:")
display(
    df.loc[review_missing, ["number_of_reviews", "last_review",
                            "reviews_per_month"]].head(10)
)

In [ ]:
for col in ["name", "host_name"]:
    print(f"\nMissingness of {col} by room type:")
    print(
        df.groupby("room_type")[col]
        .apply(lambda x: x.isna().mean())
        .round(4)
    )

**Written justification — MCAR / MAR / MNAR classification per column, and your reasoning:**

The missing values occur in name, host_name, last_review, and reviews_per_month.

name — approximately MCAR: Only a very small number of values are missing

host_name — approximately MCAR:

last_review — MAR:

reviews_per_month — MAR: Its missingness is also strongly associated with number_of_reviews = 0.

Overall, the strongest missingness pattern is in last_review and reviews_per_month, where the missing values are explained by the observed number_of_reviews column rather than occurring randomly.

In [ ]:
# TODO: apply your chosen treatment(s)
df_clean = df.copy()

# Missing listing/host names
df_clean["name"] = df_clean["name"].fillna("Unknown listing")
df_clean["host_name"] = df_clean["host_name"].fillna("Unknown host")

# For listings with zero reviews:
# no review activity means 0 reviews per month
df_clean["reviews_per_month"] = df_clean["reviews_per_month"].fillna(0)

# last_review is not useful for predicting the initial price
# and its missingness has a meaningful interpretation.
# We will remove it from the modeling features later.

print(df_clean.isnull().sum())

Written justification — why this treatment for each column, and what you'd risk with dropna() instead:

I replaced missing name and host_name values with explicit "Unknown" categories rather than deleting those rows. Only a very small number of records are affected, and deleting them would remove otherwise usable listings.

I replaced missing reviews_per_month with 0 because the missingness corresponds to listings with no reviews. In this situation, the missing value represents the absence of review activity rather than an unknown numerical measurement.

I did not attempt to invent values for last_review. A missing last review date has a meaningful interpretation for listings with no reviews, and last_review will not be used as a predictive feature.

Using dropna() on the entire dataset would unnecessarily discard around 10,000 listings because of the review-related missing values. It would also remove listings with missing names or host names even though those fields are not important for price prediction. Therefore, complete-case deletion would reduce the training data and could introduce selection bias.



---
## Part 3 — Outlier Detection & Treatment

In [ ]:
# TODO: detect outliers in at least two numeric columns
import numpy as np

def iqr_outliers(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    mask = (data[column] < lower) | (data[column] > upper)

    return {
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_count": mask.sum()
    }

price_outliers = iqr_outliers(df_clean, "price")
minimum_outliers = iqr_outliers(df_clean, "minimum_nights")

print("Price:")
print(price_outliers)

print("\nMinimum nights:")
print(minimum_outliers)

In [ ]:
display(
    df_clean.nlargest(
        20, "price"
    )[[
        "price",
        "room_type",
        "neighbourhood_group",
        "neighbourhood",
        "minimum_nights",
        "number_of_reviews"
    ]]
)


In [ ]:
display(
    df_clean.nlargest(
        20, "minimum_nights"
    )[[
        "minimum_nights",
        "price",
        "room_type",
        "neighbourhood_group",
        "number_of_reviews",
        "availability_365"
    ]]
)

In [ ]:
print("Zero-price listings:", (df_clean["price"] == 0).sum())

display(
    df_clean[df_clean["price"] == 0][[
        "price",
        "room_type",
        "neighbourhood_group",
        "minimum_nights",
        "number_of_reviews",
        "availability_365"
    ]]
)


**Written justification — for each outlier group, is it an error or a genuine listing? What evidence supports your call?**

The zero-price listings are treated as data errors for this task. A nightly Airbnb price of $0 cannot be used as a valid target for a model predicting paid nightly prices, so these rows should be removed.

The $10,000 price values are statistical outliers but should not automatically be removed. A very expensive Manhattan entire-home listing could be a genuine luxury property. The room type, neighbourhood and other listing characteristics should therefore be considered before calling such a value an error. I retain genuine high-price listings rather than imposing an arbitrary maximum price.

minimum_nights values above 365 are suspicious because they require a minimum stay longer than a full year and the dataset is intended to represent short-term rental listings. These observations should be treated as invalid/extreme duration values rather than deleting every statistical outlier based only on the IQR rule.

The important distinction is that a statistical outlier is not automatically a data error. Removing every expensive listing could eliminate legitimate Manhattan luxury properties and bias the price model toward cheaper listings.

In [ ]:
# TODO: apply treatment consistent with your judgment above
df_clean = df_clean[df_clean["price"] > 0].copy()
df_clean.loc[
    df_clean["minimum_nights"] > 365,
    "minimum_nights"
] = np.nan

print("Rows after removing zero-price listings:", len(df_clean))
print("Invalid minimum-night values remaining:",
      (df_clean["minimum_nights"] > 365).sum())

---
## Part 4 — Feature Engineering & Encoding

In [ ]:
# TODO: encode categorical columns appropriately
categorical_columns = [
    "neighbourhood_group",
    "neighbourhood",
    "room_type"
]

for col in categorical_columns:
    print(f"\n{col}:")
    print(df_clean[col].unique())

In [ ]:
# TODO: engineer at least two new features
# Feature 1: whether the listing is an entire home/apartment
df_clean["is_entire_home"] = (
    df_clean["room_type"] == "Entire home/apt"
).astype(int)

# Feature 2: whether the listing is in Manhattan
df_clean["is_manhattan"] = (
    df_clean["neighbourhood_group"] == "Manhattan"
).astype(int)

# Feature 3: log transformation of minimum nights
# Useful because minimum_nights is highly right-skewed
df_clean["log_minimum_nights"] = np.log1p(
    df_clean["minimum_nights"]
)

display(
    df_clean[
        [
            "room_type",
            "neighbourhood_group",
            "minimum_nights",
            "is_entire_home",
            "is_manhattan",
            "log_minimum_nights"
        ]
    ].head()
)

**Written justification — why these features, and which column(s) should NOT be used to predict price, and why:**

I engineered is_entire_home because an entire home/apartment normally provides more space and privacy than a private or shared room, which can influence the nightly price.

I engineered is_manhattan because Manhattan is a major geographic factor in NYC accommodation pricing. A model can also learn this information from neighbourhood_group, but the explicit binary feature provides a simple interpretable representation of the geographic effect.

I also created log_minimum_nights because minimum_nights is highly right-skewed. The logarithmic transformation reduces the influence of very large values while preserving the information contained in the feature.

The following raw columns should not be used as predictive features for a new listing: id, host_id, name and host_name because they are identifiers or free-text labels rather than stable generalizable pricing factors.

More importantly, review-related variables such as number_of_reviews, last_review and reviews_per_month, as well as availability_365, contain information that becomes available after a listing has been active. Using them to predict the initial price of a listing would create temporal leakage because that information may not exist when the price prediction is actually required.

Therefore, I exclude these post-launch variables from the price-prediction model.

---
## Part 5 — Build a Reusable Preprocessing Pipeline

In [ ]:
# TODO: train/test split (before fitting anything)
from sklearn.model_selection import train_test_split

drop_columns = [
    "id",
    "name",
    "host_id",
    "host_name",
    "last_review",
    "reviews_per_month",
    "number_of_reviews",
    "availability_365"
]
X = df_clean.drop(columns=["price"] + drop_columns)
y = df_clean["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

In [ ]:
# TODO: build Pipeline combining your cleaning, imputation, encoding, scaling steps
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Numerical features
numeric_features = [
    "latitude",
    "longitude",
    "minimum_nights",
    "is_entire_home",
    "is_manhattan",
    "log_minimum_nights",
    "calculated_host_listings_count"
]

# Categorical features
categorical_features = [
    "neighbourhood_group",
    "neighbourhood",
    "room_type"
]

# Numerical preprocessing
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

# Categorical preprocessing
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ]
)

# Combine preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

# Complete preprocessing pipeline
preprocessing_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor)
    ]
)

# Fit ONLY on training data
X_train_processed = preprocessing_pipeline.fit_transform(X_train)

# Transform test data using the already-fitted pipeline
X_test_processed = preprocessing_pipeline.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)


---
## Part 6 — Written Reflection

1.The largest preprocessing decision was handling the extreme and invalid values in price and minimum_nights. In particular, removing zero-price listings prevents the model from learning that a valid Airbnb can have a free nightly price, while retaining genuine expensive listings prevents the model from becoming biased toward ordinary low- and mid-priced properties. The impact can be evaluated by comparing model performance before and after preprocessing using the same train/test split.

2.If StayScope Analytics began receiving live listing data, the preprocessing logic would need to be applied consistently to incoming records, while all fitted statistics such as imputation medians and scaling parameters would remain those learned from the training data. The pipeline would also need monitoring for new categories, distribution changes and new types of missing or invalid values.

3.I would not delete every row with a missing or unusual value because missingness can contain meaningful information and statistical outliers can represent genuine Airbnb listings. Deleting all such rows would unnecessarily reduce the dataset and could introduce bias, whereas targeted treatment allows valid information to be retained while correcting genuinely invalid observations.